# Winter wheat phenology — TorchCrop against CyBench

**What this notebook does.** It derives the simulated stage dates from the
10 km TorchCrop run, averages them to national dates, and compares them with
the CyBench crop calendar. RMSE, MAE, Bias and Pearson r are reported per stage
and per country.

## What each side actually contains

Read this before the results — two of the three stage pairings are proxies, and
the fourth requested stage does not exist on either side.

| Stage | TorchCrop | CyBench | Comparable? |
|---|---|---|---|
| Sowing | **constant input**, DOY 270 for every cell | `sos` | weakly — see below |
| Flowering | **not in the run** | **not in the calendar** | no |
| Maturity | first day at `DVS ≥ 2` | `eos` | yes, best pairing |
| Harvest | not modelled; = maturity + `HARVEST_LAG_DAYS` (0) | `eos` | as maturity |

* **Sowing is not a prediction.** The SIMPLACE export carries no sowing
  calendar, so `cropmodelling4eu.torchcrop.run` sows every cell from Crete to Lapland on
  DOY 270. Its only degree of freedom is one continental offset.
* **CyBench `sos` is not a sowing date, and it changes meaning with climate.**
  It is ~DOY 330–360 in Spain, Greece and Portugal (autumn sowing) but ~DOY
  40–75 in Germany, Poland and the Baltics (post-winter green-up). The northern
  residuals are a convention difference, not a model error; §7 separates the
  two regimes so this is visible rather than buried in a mean.
* **Flowering is absent from both.** `run_batch` discards the DVS trajectory
  once it has dated the fertilizer schedule, so no anthesis date reaches the
  Parquet, and the CyBench calendar carries only a season start and end. It is
  therefore not evaluated here. Emitting the `DVS ≥ 1` crossing in
  `cropmodelling4eu.torchcrop.run` and re-running the shards would supply the
  simulated side; the reference side would still need another source.
* **Harvest is maturity.** LINTUL-5 stops the crop at `DVS = 2` and models no
  drydown. The row is kept because the assumption should be visible, not
  because it adds information; raise `config.HARVEST_LAG_DAYS` to test a lag.

## Two conventions

1. **Error is `simulated − observed`**, so a positive bias means the model is
   **late**.
2. **Every date statistic is circular.** A day-of-year lives on a circle:
   means go through `doy.circular_mean_doy`, errors through
   `doy.doy_difference`, and correlations are computed on both sides unwrapped
   about the observed circular mean. Without that, Spain's regional `sos` of
   DOY 363 and DOY 0.7 average to July.

All reusable code is in [`utils/`](utils/); this notebook is the workflow only.

## 0. Setup

In [ ]:
import logging
import sys
from pathlib import Path

# cropmodelling4eu is installed (pip install -e .), so the evaluation
# library is imported like any other package rather than off sys.path.

import numpy as np
import pandas as pd

from cropmodelling4eu.evaluation import aggregate, config, cybench, doy, metrics, plots, regions, torchcrop
from cropmodelling4eu.evaluation.style import use_style

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s",
                    force=True)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

PALETTE = use_style("light")
config.ensure_output_dirs()

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 140)

print(f"TorchCrop run : {config.TORCHCROP_RUN_DIR}")
print(f"CyBench root  : {config.CYBENCH_ROOT}")
print(f"Outputs       : {config.OUTPUT_DIR}")

## 1. Scope — which countries are evaluable

In [ ]:
COUNTRIES = config.available_countries()

scope = pd.DataFrame({
    "code": list(config.TARGET_COUNTRIES),
    "name": [config.COUNTRY_NAMES.get(c, c) for c in config.TARGET_COUNTRIES],
    "eu27": [c in config.EU27 for c in config.TARGET_COUNTRIES],
    "schengen_non_eu": [c in config.SCHENGEN_NON_EU for c in config.TARGET_COUNTRIES],
    "in_cybench": [c in COUNTRIES for c in config.TARGET_COUNTRIES],
}).set_index("code")

print(f"{len(COUNTRIES)} of {len(config.TARGET_COUNTRIES)} target countries are evaluable")
print("missing:", ", ".join(scope.index[~scope["in_cybench"]]))
scope

## 2. Load the simulation and derive the stage dates

`days_to_maturity` counts from the sowing latch, so maturity is
`sowing_doy + days_to_maturity` wrapped back into the year: a crop sown on
DOY 270 and maturing 272 days later matures on DOY 177 of the following
calendar year, which is the harvest year the row is labelled with.

In [ ]:
sim = torchcrop.load_simulation(columns=[
    "SimplaceID", "year", "lon", "lat", "days_to_maturity", "final_dvs", "yield_t_ha",
])

print("stages derived:", ", ".join(torchcrop.PHENOLOGY_COLUMNS))
for stage in config.STAGES:
    print(f"  {stage.label:9s} <- {stage.sim_col:14s} vs CyBench '{stage.obs_col}'")

sim[list(torchcrop.PHENOLOGY_COLUMNS)].describe().T

## 3. Assign each 10 km cell to a country

In [ ]:
cells = torchcrop.simulation_cells(sim)
cells = regions.assign_cells_to_countries(
    cells, COUNTRIES, cache=regions.default_cache_path(len(COUNTRIES))
)

sim = sim.merge(cells[["SimplaceID", "country", "snapped"]], on="SimplaceID", how="left")
scoped = sim[sim["country"].notna()].copy()

print(f"{cells['country'].notna().sum():,} of {len(cells):,} cells inside the "
      f"CyBench footprint ({cells['snapped'].sum():,} matched by the {config.SNAP_KM} km snap)")
print(f"{len(scoped):,} of {len(sim):,} simulated cell-seasons kept")

cells_per_country = (
    cells.dropna(subset=["country"]).groupby("country").size().rename("cells").to_frame()
)
cells_per_country["share_%"] = (
    100 * cells_per_country["cells"] / cells_per_country["cells"].sum()
).round(1)
cells_per_country.T

## 4. Aggregate the simulation to national dates

Every stage column is flagged circular, so the national date is a circular
mean. `season_length_days` is a **duration**, not a date, so it stays linear.

In [ ]:
STAGE_COLS = {stage.sim_col: True for stage in config.STAGES}
STAGE_COLS["season_length_days"] = False

sim_country = aggregate.aggregate_simulated(scoped, STAGE_COLS)
sim_country.head()

## 5. Load and aggregate the CyBench crop calendar

The calendar is **static** — one `sos`/`eos` per administrative unit, with no
year dimension. Every simulated season is therefore compared against the same
reference date, which means interannual scatter in the simulation can only be
penalised, never rewarded. Regions are weighted by `crop_area` from the CyBench
crop mask, so a 200 ha alpine district does not outvote a 200 000 ha plain.

In [ ]:
calendar = cybench.load_calendar(COUNTRIES)
crop_mask = cybench.load_crop_mask(COUNTRIES)
obs_country = aggregate.aggregate_observed_calendar(calendar, crop_mask)

obs_country["sos_date"] = obs_country["sos"].map(lambda d: doy.doy_to_month_day(d))
obs_country["eos_date"] = obs_country["eos"].map(lambda d: doy.doy_to_month_day(d))
obs_country.set_index("country").round(1)

### The two `sos` conventions

The split is the reason a single "sowing" metric would be meaningless. A
country whose observed `sos` falls in the autumn is reporting a sowing date; one
whose `sos` falls after New Year is reporting a green-up.

In [ ]:
AUTUMN_SOS_THRESHOLD = 200.0  # DOY; above it, sos is an autumn date

obs_country["sos_regime"] = np.where(
    obs_country["sos"] >= AUTUMN_SOS_THRESHOLD, "autumn sowing", "post-winter green-up"
)
regimes = obs_country.groupby("sos_regime")["country"].apply(list)
for regime, members in regimes.items():
    print(f"{regime:22s} ({len(members):2d}): {', '.join(sorted(members))}")

## 6. Pair the two sides

The join is on `country` alone: the reference has no year, so each simulated
country-year is paired against its country's climatological calendar.

In [ ]:
paired = aggregate.pair_observations(sim_country, obs_country, ["country"])

for stage in config.STAGES:
    paired[f"{stage.key}_error"] = doy.doy_difference(
        paired[stage.sim_col], paired[stage.obs_col]
    )

print(f"{len(paired)} country-years across {paired['country'].nunique()} countries")
paired.head()

## 7. Metrics by stage

Pooled across every country-year. Two correlations are undefined here and come
back as NaN, both for the same reason — a constant has no variance:

* **Sowing**, because the *simulated* date is the DOY 270 latch.
* **Every per-country row** in the next section, because the *observed* date is
  a single climatological value repeated across that country's years.

The pooled maturity `pearson_r` is therefore a **spatial** correlation across
the 23 countries, not an interannual one.

In [ ]:
pooled = {}
for stage in config.STAGES:
    pooled[stage.label] = metrics.phenology_metrics(
        paired[stage.obs_col], paired[stage.sim_col]
    )

pooled_table = pd.DataFrame(pooled).T[list(metrics.PHENOLOGY_METRIC_ORDER)]
pooled_table["observed date"] = pooled_table["obs_mean"].map(doy.doy_to_month_day)
pooled_table["simulated date"] = pooled_table["sim_mean"].map(doy.doy_to_month_day)

pooled_table.round(2)

In [ ]:
for stage in config.STAGES:
    print(f"--- {stage.label} ---")
    print(stage.caveat)
    print()

### Per country and stage

One table per stage, and a wide summary across all three.

In [ ]:
stage_metrics = {
    stage.key: metrics.metrics_by_group(
        paired, stage.obs_col, stage.sim_col, metric_fn=metrics.phenology_metrics
    )
    for stage in config.STAGES
}

summary = pd.concat(
    {stage.label: stage_metrics[stage.key][["n", "bias", "mae", "rmse", "pearson_r"]]
     for stage in config.STAGES},
    axis=1,
)
summary.to_csv(config.TABLE_DIR / "phenology_metrics_by_country_stage.csv",
               float_format="%.3f")
summary.round(1)

> **The per-country `pearson_r` column is NaN by construction, not by
> accident.** The CyBench calendar has no year dimension, so within one country
> the observed date is the same value in every paired year — a constant has no
> variance to correlate against. The column is kept rather than hidden so the
> limitation is visible in the table itself.
>
> The **pooled** `pearson_r` in §7 is therefore a *spatial* correlation across
> the 23 countries, not an interannual one: it says the model orders the
> countries' seasons correctly. Nothing here tests whether it tracks a warm
> year against a cold one — that would need a year-resolved reference.

In [ ]:
maturity_ranked = metrics.rank_countries(stage_metrics["maturity"], by="rmse")

plots.metric_table(
    maturity_ranked, ("rank", *metrics.PHENOLOGY_METRIC_ORDER), precision=1,
    caption="Maturity (simulated DVS >= 2) against CyBench 'eos', ranked by "
            "RMSE in days. obs_mean/sim_mean are days-of-year; bias, MAE and "
            "RMSE are in days, positive meaning the model is late.",
)

### Sowing, split by `sos` convention

Pooling the two regimes would report a mean of two incompatible quantities.
Split, the autumn-sowing countries give a usable check on the DOY 270 latch;
the green-up countries do not, and their column is here only to make the size of
the convention gap explicit.

In [ ]:
# `sos_regime` is already on `paired`: it was added to obs_country before the
# join, so it came across with the rest of the reference columns.
sowing_rows = []
for regime, block in paired.groupby("sos_regime"):
    row = metrics.phenology_metrics(block["sos"], block["sowing_doy"])
    row["countries"] = block["country"].nunique()
    row["regime"] = regime
    sowing_rows.append(row)

sowing_table = pd.DataFrame(sowing_rows).set_index("regime")
sowing_table["observed date"] = sowing_table["obs_mean"].map(doy.doy_to_month_day)
sowing_table["simulated date"] = sowing_table["sim_mean"].map(doy.doy_to_month_day)
sowing_table[["countries", "n", "observed date", "simulated date", "bias", "mae", "rmse"]].round(1)

## 8. Figures

### 8.1 Observed against simulated, one panel per stage

Both axes are days-of-year unwrapped about the observed circular mean, then
relabelled as calendar dates — a scatter axis is linear, but the underlying
quantity is not.

In [ ]:
for stage in config.STAGES:
    fig = plots.scatter_one_to_one(
        paired, stage.obs_col, stage.sim_col, circular=True,
        stats=pooled[stage.label],
        metric_keys=("n", "bias", "rmse", "pearson_r"),
        title=f"{stage.label}: simulated against CyBench '{stage.obs_col}'",
        xlabel=f"CyBench {stage.obs_col}",
        ylabel=f"TorchCrop {stage.label.lower()}",
    )
    plots.save(fig, f"phenology_01_scatter_{stage.key}")
    display(fig)

### 8.2 Per country, for the maturity pairing

Only maturity gets the small-multiple treatment: it is the one pairing where
both sides mean the same thing, so it is the one where a per-country panel says
something about the model.

In [ ]:
fig = plots.scatter_small_multiples(
    paired, "eos", "maturity_doy", metrics=stage_metrics["maturity"],
    order=[c for c in maturity_ranked.index if c != "ALL"],
    circular=True, annotate=("bias", "rmse"),
    title="Maturity against CyBench 'eos', by country (ordered by RMSE)",
    xlabel="CyBench end of season",
    ylabel="TorchCrop maturity",
)
plots.save(fig, "phenology_02_maturity_small_multiples")
fig

### 8.3 Country-wise bias, one figure per stage

Positive (warm) means the model is **late**.

In [ ]:
for stage in config.STAGES:
    fig = plots.bias_bars(
        metrics.rank_countries(stage_metrics[stage.key], by="bias"),
        value_col="bias", error_col="rmse",
        title=f"{stage.label} bias by country (simulated - observed)",
        xlabel="Bias (days; positive = simulated late)",
    )
    plots.save(fig, f"phenology_03_bias_{stage.key}")
    display(fig)

### 8.4 Maps

In [ ]:
polygons = regions.load_country_polygons(COUNTRIES)

for stage in config.STAGES:
    fig = plots.country_choropleth(
        polygons, stage_metrics[stage.key]["bias"],
        title=f"{stage.label} bias (simulated - observed)",
        cbar_label="Bias (days; positive = late)", diverging=True,
    )
    plots.save(fig, f"phenology_04_map_bias_{stage.key}")
    display(fig)

In [ ]:
cell_maturity = (
    scoped.groupby(["SimplaceID", "lon", "lat"], as_index=False)
    .agg(days_to_maturity=("days_to_maturity", "mean"))
)

fig = plots.cell_map(
    cell_maturity, "days_to_maturity",
    title=f"Simulated season length, sowing (DOY {config.SIM_SOWING_DOY}) to maturity",
    cbar_label="Days", overlay=polygons,
)
plots.save(fig, "phenology_05_map_season_length")
fig

## 9. Summary

In [ ]:
paired.to_csv(config.TABLE_DIR / "phenology_paired_country_year.csv",
              index=False, float_format="%.4f")
pooled_table.to_csv(config.TABLE_DIR / "phenology_metrics_pooled.csv",
                    float_format="%.3f")
print(f"tables written to {config.TABLE_DIR}")
print(f"figures written to {config.FIGURE_DIR}")

pooled_table[["n", "observed date", "simulated date", "bias", "mae", "rmse",
              "pearson_r"]].round(1)

### What the numbers mean, and what they do not

* **Maturity is the only result to quote.** Both sides mean the same thing
  there, and the correlation across countries is real: the model orders the
  countries' seasons correctly even where it places them wrongly in absolute
  terms.
* **The maturity bias is not purely a model error.** CyBench `eos` is an end of
  season, which falls at or after harvest; simulated maturity is `DVS = 2`. Part
  of the gap is the maturity-to-harvest interval that LINTUL-5 does not model.
* **Sowing measures the DOY 270 assumption, not the model.** For the autumn-`sos`
  countries the bias is a real statement about that constant. For the
  green-up-`sos` countries it is a statement about the CyBench convention and
  should not be read as skill.
* **Harvest carries no information beyond maturity** while
  `config.HARVEST_LAG_DAYS` is 0.
* **The reference is climatological.** With no year dimension on `sos`/`eos`,
  none of these metrics test whether the model tracks a warm or a cold season —
  only whether it places the average season correctly.